# MACHINE LEARNING PROJECT

### Track T4 – Semi-Supervised Learning (SSL)
The objective of the project is to study and apply semi-supervised learning techniques to tabular data, analyzing how the limited availability of labels affects the performance of predictive models.

I'm interested in cybersecurity, so I chose the CSE-CIC-IDS2018 Intrusion CSVs (IDS 2018) dataset for the classification problem.

The dataset is based on logs from university servers, which recorded various DoS attacks during the publicly available period.
In total, there are eighty columns within this dataset, each of which corresponds to an entry in the IDS logging system that the Unversity of New Brunswick has in place.

Each entry in the dataset is originally labeled with multiple classes, such as Benign, FTP-BruteForce, and Other. To simplify the task and formulate it as a binary classification problem, I will consolidate all non-Benign labels into a single Malicious category. This way, the model will learn to distinguish between Benign and Malicious network traffic.

## Import libraries and models
I import the libraries and modules that we will need for the project.

In [ ]:
import pandas as pd
import dask.dataframe as dd
import os

## Load and manipulate the dataset
I load the dataset using only a single large CSV instead of all the files.This simplifies processing, avoids mismatched type issues, and reduces memory usage.  
At the same time, I convert all labels that are not "Benign" into a single "Malicious" category.  
I use Dask that is a library which performs lazy loading to avoid loading the entire dataset into memory.

In [ ]:
dataset = dd.read_csv('Dataset/02-20-2018.csv', assume_missing = True)
dataset['Label'] = dataset['Label'].map(lambda x: 'Malicious' if x != 'Benign' else 'Benign')
dataset["Label"].unique().compute()

Now I check the entire dataset for entries with any missing feature values.

In [ ]:
def printStats():
    dirty_entries_dt = dataset[dataset.isna().any(axis=1)]
    dirty_entries = dirty_entries_dt.shape[0].compute()
    dirty_benign = dirty_entries_dt[dirty_entries_dt["Label"] == "Benign"].shape[0].compute()
    dirty_malicious = dirty_entries_dt[dirty_entries_dt["Label"] == "Malicious"].shape[0].compute()
    dirty_not_labeled = dirty_entries_dt[dirty_entries_dt["Label"].isna()].shape[0].compute()
    totalRows = dataset.shape[0].compute()
    print(f"Entries that have at least 1 null feature value: {dirty_entries} ; of those Benign: {dirty_benign} , Malicious: {dirty_malicious} , Not_Labeled: {dirty_not_labeled} , Other: {dirty_entries-dirty_benign-dirty_malicious-dirty_not_labeled}\nTotal Entries: {totalRows}\nPercentage of dirty entries: {dirty_entries / totalRows:.2%}")

printStats()

So i check the balance of the two labels

In [ ]:
benign_entries = dataset[dataset["Label"] == "Benign"].shape[0].compute()
malicious_entries = dataset[dataset["Label"] == "Malicious"].shape[0].compute()
print(f"Benign Entries of the dataset: {benign_entries}\nMalicious Entries of the dataset: {malicious_entries}")

At this point, we see that entries with missing values are less than 1% of the total dataset. Dropping features or filling missing values doesn’t make sense, since every feature is important in traffic analysis, and assigning values could introduce noise and break the real patterns. Therefore, I will remove these incomplete entries from the dataset. Moreover, as noted above, the dataset is heavily imbalanced towards Benign entries (there are many more of them), so removing the dirty entries, which are mostly Benign, is even more justified.

In [ ]:
dataset = dataset.dropna()
printStats()

## Features Cleaning
This dataset contains network traffic analysis data between devices and university servers, so it is necessary to carefully examine all the features available in each entry and remove those that are not relevant for analyzing future network traffic and making predictions, thereby essentially avoiding model overfitting.

In [ ]:
print(dataset.columns)